## HTTP Layer — `curl`

**Support context:** `nc -zv` proves the **TCP port** accepts connections. It doesn't tell you whether the **app** behind it works. `curl` speaks HTTP, so it answers the next question: **"when my request arrives, what does the app say back?"** This is Step 4 of the ticket checklist. Most "the site is down" tickets turn out to be one of four things: **DNS, connection, TLS, or the app returning an error**. `curl -v` shows you which one in a single command.

**Developer bridge:** curl is `requests.get()` from the shell.
- `-v` is like logging the full request and response.
- `-I` is `requests.head()`.
- `-w` is like wrapping the call in timing code.
- The status codes are the ones your Django/FastAPI apps return behind gunicorn and nginx: 404, 500, 502.

**COMMAND:** `curl`

**PURPOSE**:   Prove the HTTP layer works (status code, headers, TLS, timing)

**SYNTAX:** `curl [flags] URL`

| Flag | Meaning | When you reach for it |
|---|---|---|
| `-v` | Verbose: DNS, connect, TLS, request and response headers | First move on any "site broken" ticket |
| `-s` | Silent: no progress meter | Scripts. `-sv` keeps the verbose lines without the progress meter. |
| `-I` | HEAD request, headers only | Quick status check without the HTML |
| `-i` | Include response headers **and** the body | When you want to see both |
| `-L` | Follow redirects (301/302) | When `-I` shows a 301 and you need the final page |
| `-o /dev/null` | Throw the body away | Paired with `-w` when you only want metrics |
| `-w '...'` | Print metrics after the request | Status code and timing breakdown |
| `-k` | Skip TLS cert verification | **Diagnosis only.** Proves the cert is the problem. |
| `-H 'Host: x'` | Set a request header | Test a vhost by IP without changing DNS |
| `--resolve host:port:IP` | Force a hostname to an IP for this one request | Bypass DNS, like a one-off `/etc/hosts` entry |
| `-m N` / `--connect-timeout N` | Overall / connect-phase time limit | Stop waiting on blackholed hosts (your last ticket) |
| `-f` | Non-zero exit code on HTTP ≥ 400 | Scripts and health checks |

**EXAMPLE 1 — read a verbose request:**
```bash
curl -sv https://example.com -o /dev/null
```
Real output from your machine (certificate-chain and HTTP/2 stream lines trimmed):

In [ ]:
$ curl -sv https://example.com -o /dev/null
* Host example.com:443 was resolved.
* IPv6: 2606:4700:10::ac42:93f3, 2606:4700:10::6814:179a
* IPv4: 172.66.147.243, 104.20.23.154
*   Trying 172.66.147.243:443...
* Connected to example.com (172.66.147.243) port 443
* ALPN: curl offers h2,http/1.1
*  CAfile: /etc/ssl/certs/ca-certificates.crt
* SSL connection using TLSv1.3 / TLS_AES_256_GCM_SHA384 / X25519 / id-ecPublicKey
* ALPN: server accepted h2
* Server certificate:
*  subject: CN=example.com
*  start date: Sep 26 22:49:11 2026 GMT
*  expire date: Dec 25 22:56:35 2026 GMT
*  subjectAltName: host "example.com" matched cert's "example.com"
*  issuer: C=US; O=SSL Corporation; CN=Cloudflare TLS Issuing ECC CA 3
*  SSL certificate verify ok.
> GET / HTTP/2
> Host: example.com
> User-Agent: curl/8.5.0
> Accept: */*
>
< HTTP/2 200
< date: Wed, 11 Nov 2026 19:04:03 GMT
< content-type: text/html; charset=utf-8
< server: cloudflare
< cf-cache-status: HIT
<
* Connection #0 to host example.com left intact

**READ THE OUTPUT:** there are three line prefixes, and each is a layer you can check.

| Prefix | Who wrote it | What it proves |
|---|---|---|
| `*` | curl itself: resolve, connect, TLS | `was resolved` = DNS OK. `Connected to` = TCP OK (what `nc` proved). `SSL certificate verify ok` = TLS OK. **If it fails here, the app never saw your request.** |
| `>` | Your request | Method, path, and `Host:` header. Check `Host:` when testing vhosts. |
| `<` | The server's reply | Status line + headers. **If you got this far, the network is fine. The problem is in the app or the proxy.** |

Lines worth reading on every ticket:
- `expire date` — an expired cert is a classic ticket.
- `subjectAltName ... matched` — if the name doesn't match, curl fails with exit 60.
- `server: cloudflare` + `cf-cache-status` — a CDN sits in front, so the response may never have touched the origin server.

**EXAMPLE 2 — your own server, three outcomes (two terminals):**

Terminal A is the "service":
```bash
mkdir -p ~/curltest && cd ~/curltest && echo "hello from dennie" > index.html
python3 -m http.server 8000 --bind 127.0.0.1
```
Terminal B is the "client":
```bash
curl -i http://localhost:8000/        # 200 + body
curl -I http://localhost:8000/nope    # 404: server UP, path wrong
```

In [ ]:
$ curl -i http://localhost:8000/
HTTP/1.0 200 OK
Server: SimpleHTTP/0.6 Python/3.12.3
Date: Wed, 11 Nov 2026 19:04:04 GMT
Content-type: text/html
Content-Length: 18
Last-Modified: Wed, 11 Nov 2026 19:04:03 GMT

hello from dennie

$ curl -I http://localhost:8000/nope
HTTP/1.0 404 File not found
Server: SimpleHTTP/0.6 Python/3.12.3
Date: Wed, 11 Nov 2026 19:04:04 GMT
Connection: close
Content-Type: text/html;charset=utf-8
Content-Length: 335

Now **Ctrl+C in Terminal A** to stop the server, then in Terminal B:
```bash
curl -sv http://localhost:8000/ ; echo "exit code: $?"
```

In [ ]:
$ curl -sv http://localhost:8000/ ; echo "exit code: $?"
* Host localhost:8000 was resolved.
* IPv6: ::1
* IPv4: 127.0.0.1
*   Trying [::1]:8000...
* connect to ::1 port 8000 from ::1 port 41496 failed: Connection refused
*   Trying 127.0.0.1:8000...
* connect to 127.0.0.1 port 8000 from 127.0.0.1 port 40042 failed: Connection refused
* Failed to connect to localhost port 8000 after 0 ms: Couldn't connect to server
* Closing connection
exit code: 7

**READ THE OUTPUT:**
- **404 vs refused.** The 404 has `<` lines: the server answered and said "no such path". The refused run has **no `<` lines at all**: nothing was listening.
- **`after 0 ms`** — refused is instant. A firewall drop (your last ticket) hangs until the timeout. Same rule as `nc`: refused is fast, a timeout is slow.
- **`localhost` tried `::1` first, then `127.0.0.1`.** `localhost` resolves to both. The server was bound to `127.0.0.1` only, so even while it was running, the `::1` attempt would be refused and curl would fall back to IPv4. In a real ticket, "works with `127.0.0.1`, fails with `localhost`" points straight at this.
- **Exit code 7** = couldn't connect. That maps to the `ss -tlnp` lesson: what is listening, and on which address.

**EXAMPLE 3 — timing breakdown (where is it slow?):**
```bash
curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://example.com
```

In [ ]:
$ curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://example.com
code=200 dns=0.060713 connect=0.094789 tls=0.149605 ttfb=0.225108 total=0.225214

**READ THE OUTPUT:** each value is a **timestamp since start**, not a duration. Subtract to get each phase:

| Phase | Calculation | Your run | Big value means |
|---|---|---|---|
| DNS | `dns` | 61 ms | Resolver problem → `dig`, `/etc/resolv.conf` |
| TCP connect | `connect - dns` | 34 ms | Network latency or packet loss |
| TLS handshake | `tls - connect` | 55 ms | Slow TLS, often a distant server or an overloaded LB |
| App thinking | `ttfb - tls` | 75 ms | **The app is slow**: a slow DB query, gunicorn workers saturated |
| Download | `total - ttfb` | ~0 ms | Big body or slow link |

A server-side error, for comparison:

In [ ]:
$ curl -s -o /dev/null -w '%{http_code}\n' https://httpbin.org/status/503
503

**curl exit codes = the layer that failed.** Memorize these. They tell you where it broke before you've read a single line of `-v` output.

| Exit | Meaning | Layer → next command |
|---|---|---|
| 6 | Couldn't resolve host | DNS → `dig`, `/etc/resolv.conf` |
| 7 | Couldn't connect (refused) | Nothing listening → `ss -tlnp` on the server |
| 28 | Timeout | Firewall drop / routing → `ping`, `traceroute`, ufw |
| 35 / 60 | TLS handshake / cert verify failed | Cert expired, wrong name, missing CA |
| 52 | Empty reply from server | App crashed mid-request |
| 56 | Connection reset | Proxy/LB killed the connection |
| 0 + `4xx`/`5xx` | HTTP worked, the app said no | App logs → `journalctl -u nginx` |

**Key point:** exit code **0 with a 500** is still a success from curl's point of view, because the HTTP exchange completed. Add `-f` if a script needs to treat that as a failure.

Exit 6 on your machine:

In [ ]:
$ curl -s https://nosuchhost.invalid; echo "exit: $?"
exit: 6

**HTTP status codes you'll meet on tickets:**

| Code | Meaning | Support read |
|---|---|---|
| 200 | OK | Works from here. Ask what the user sees differently (path, auth, location, cache). |
| 301 / 302 | Redirect | Check `location:`. A loop shows the same URL repeating under `-L -v`. |
| 401 / 403 | Unauthorized / Forbidden | Auth or permissions, not an outage |
| 404 | Not found | Server up, wrong path or vhost |
| 500 | Internal server error | App exception → app logs |
| 502 | Bad gateway | nginx is up, the **backend** (gunicorn) isn't answering |
| 503 | Service unavailable | Overloaded, or in maintenance |
| 504 | Gateway timeout | Backend reachable but too slow |

**TICKET SCENARIO:** *"Customers get an error page on https://app.acme.io, but our monitoring says the server is up."*

**DIAGNOSIS PATH:**
1. `curl -sI https://app.acme.io`. **502** → nginx up, backend down. **503** → overloaded or maintenance. **504** → backend too slow.
2. `curl -sv https://app.acme.io -o /dev/null`. Does TLS succeed? Does the `server:` header match what you expect, or is there a CDN or LB in front?
3. On the server: `curl -I http://127.0.0.1:8000` hits gunicorn directly and **skips nginx**. If it returns 200, the problem is in the nginx config or the upstream address.
4. `curl --resolve app.acme.io:443:10.0.0.12 https://app.acme.io` tests one specific backend node, bypassing DNS and the LB.

**WHAT TO LOOK FOR:**
- `* Trying ...` with nothing after it → timeout (network or firewall)
- `Connection refused` → service down, or bound to a different address
- `SSL certificate problem` → cert issue. Confirm with `-k`, but don't ship `-k` as the fix.
- `< HTTP/1.1 301` + `location:` → follow with `-L -v` and watch for a loop

**GOTCHAS**
- **`-k` is a diagnostic, not a fix.** If `-k` works, you've proven the cert is the problem. Now go fix the cert.
- **Proxy env vars.** `env | grep -i proxy`. A stale `https_proxy` exported in `.bashrc` makes curl fail while the browser works, which is common on corporate laptops. Use `curl --noproxy '*'` to test without the proxy.
- **`-I` sends HEAD, not GET.** Some apps answer HEAD differently (405, or a different code). If the results look odd, use `curl -s -o /dev/null -w '%{http_code}'`, which sends a real GET.
- **`-v` with `-o` shows a progress meter** mixed into the output. Use `-sv` to keep the verbose lines and drop the meter.
- **WSL:** a server bound to `127.0.0.1` inside WSL2 is usually still reachable from Windows at `localhost:8000`, thanks to WSL's localhost forwarding. That's a WSL convenience; on real Linux, `127.0.0.1` means this machine only.
- **WSL / Windows:** in Windows PowerShell 5.1, `curl` is an alias for `Invoke-WebRequest`. Use `curl.exe` there to get the real curl.

### TRY NOW

1. Restart the Python server with `--bind 0.0.0.0` instead of `127.0.0.1`. Run `ss -tlnp | grep 8000` and compare the local address with a `127.0.0.1` run. This previews the `ss` lesson.

Terminal A, to start the server bound to localhost only:

In [ ]:
# terminal A
mkdir -p ~/curltest && cd ~/curltest && echo "hello from dennie" > index.html

python3 -m http.server 8000 --bind 127.0.0.1
Serving HTTP on 127.0.0.1 port 8000 (http://127.0.0.1:8000/) ...

Terminal B, to see what's listening:

In [ ]:
# 1. paste ss -tlnp output for both binds here

# terminal B
ss -tlnp | grep 8000
LISTEN 0      5           127.0.0.1:8000       0.0.0.0:*    users:(("python3",pid=24501,fd=3))

In [ ]:
# terminal A
python3 -m http.server 8000 --bind 0.0.0.0
Serving HTTP on 0.0.0.0 port 8000 (http://0.0.0.0:8000/) ...

In [ ]:
# terminal B
ss -tlnp | grep 8000
LISTEN 0      5             0.0.0.0:8000       0.0.0.0:*    users:(("python3",pid=24979,fd=3))

**Local Address:Port** column. It shows `127.0.0.1:8000` for the first run and `0.0.0.0:8000` for the second.
You don't need sudo here. You can see `python3` in the process column because the server runs as you, `dennie`. For other users' processes you'd need `sudo ss -tlnp`.

2. Find the slow phase. Run the `-w` timing command against `https://httpbin.org/delay/3`. Which phase takes the 3 seconds, and what does that tell you?

In [ ]:
# 2. paste httpbin.org/delay/3 timing output here

curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://httpbin.org/delay/3
code=200 dns=0.199011 connect=0.346441 tls=0.560324 ttfb=3.691480 total=3.691583

Subtract each value from the one after it, like in the notebook's timing table, and find which gap is about 3 seconds.

3. A blcackholed host. Run `curl -m 3 http://10.255.255.1 ; echo $?`. Which exit code does a blackholed host give you?

In [ ]:
# 3. paste curl -m 3 http://10.255.255.1 output + exit code here

time curl -m 3 http://10.255.255.1 ; echo "exit code: $?"
curl: (28) Connection timed out after 3001 milliseconds

real    0m3.095s
user    0m0.024s
sys     0m0.019s
exit code: 28

Added `time` so you can check that it's a slow failure, like the `real 3.1s` you got in the `nc` lesson. Note which exit code you get and compare it with the 7 from connection refused.

### DRILL

**Q1.** `curl -sv https://shop.example` prints `* Trying 203.0.113.10:443...` and then sits there until `-m` expires. Which exit code do you get, and which layer failed?

**Q2.** `curl -sI https://shop.example` returns `HTTP/1.1 502 Bad Gateway` with `server: nginx`. Is the server down? What do you check next, and on which machine?

**Q3.** curl fails with exit 60, and `curl -k` works. What have you proven, and why is "just add `-k`" the wrong fix to hand the customer?

**Q4.** In your timing output `ttfb - tls` is 4.8 s and every other phase is under 100 ms. Network team or app team?

### DRILL ANSWERS

*(fill in after the drill)*

**Theory follow-up:** freeCodeCamp Computer Networking course, **"FTP, HTTP, ARP & ICMP" — 12:01:21**. Covers the HTTP request/response model that the `>` and `<` lines show you.